# DisappearingPairs

[Codility task](https://app.codility.com/programmers/trainings/4/disappearing_pairs/) - Exercise 4, Algorithmic skills

A string over `{A, B, C}` can be transformed by removing one occurrence of `"AA"`, `"BB"` or `"CC"`. Repeat while some rule applies. `solution(S)` returns a string that can result from such a sequence.

| S | result |
|---|--------|
| `"ACCAABBC"` | `"AC"` (`ACCAABBC -> AAABBC -> ABBC -> AC`) |
| `"ABCBBCBA"` | `""` (`ABCBBCBA -> ABCCBA -> ABBA -> AA -> ""`) |
| `"BABABA"` | `"BABABA"` (nothing to remove) |

Assumptions: length of `S` in `[0..50,000]`, only the letters `A`, `B`, `C`.

## Idea: a stack

Scan left to right. If the new letter equals the top of the stack, the two cancel (pop); otherwise push it. The stack never holds two equal adjacent letters, so when the scan ends no rule applies.

Time O(N), space O(N).

In [ ]:
def solution(S):
    stack = []
    for letter in S:
        if stack and stack[-1] == letter:
            stack.pop()
        else:
            stack.append(letter)
    return "".join(stack)

## Official examples

In [ ]:
assert solution("ACCAABBC") == "AC"
assert solution("ABCBBCBA") == ""
assert solution("BABABA") == "BABABA"
print("examples OK")

## Edge cases

In [ ]:
assert solution("") == ""
assert solution("A") == "A"
assert solution("AA") == ""
assert solution("AAA") == "A"
assert solution("AABB") == ""
assert solution("ABBA") == ""                 # removing BB exposes AA
assert solution("ABCCBA") == ""               # nested pairs
assert solution("ABC") == "ABC"
print("edge cases OK")

## Is the result really unique?

The task says "any" result, but the answer is **unique**: whichever removal order you choose, you end at the same string. The cell below proves it by brute force for every string over `{A, B, C}` of length up to 8: it explores **all** possible orders of removals and checks that exactly one terminal string exists and that it equals the stack result.

In [ ]:
import itertools
from functools import lru_cache


@lru_cache(maxsize=None)
def terminals(s):
    """Set of all strings reachable by removals from which no rule can be applied."""
    results = set()
    found = False
    for i in range(len(s) - 1):
        if s[i] == s[i + 1]:
            found = True
            results |= terminals(s[:i] + s[i + 2:])
    return results if found else {s}


checked = 0
for n in range(0, 9):
    for letters in itertools.product("ABC", repeat=n):
        s = "".join(letters)
        reachable = terminals(s)
        assert len(reachable) == 1, (s, reachable)         # confluence: only one possible end result
        assert reachable == {solution(s)}, s
        checked += 1
print(f"checked {checked} strings: always a single terminal string, equal to the stack result")

## Large inputs

In [ ]:
import random
import time


def irreducible(s):
    return "AA" not in s and "BB" not in s and "CC" not in s


rng = random.Random(6)
N = 50_000
cases = {
    "all A (even)": "A" * N,
    "all A (odd)": "A" * (N - 1),
    "alternating ABAB...": "AB" * (N // 2),
    "random": "".join(rng.choice("ABC") for _ in range(N)),
    "nested pairs": "".join("ABC"[i % 3] for i in range(N // 2)) + "".join("ABC"[i % 3] for i in reversed(range(N // 2))),
}
for name, s in cases.items():
    start = time.perf_counter()
    result = solution(s)
    assert irreducible(result)
    assert (len(s) - len(result)) % 2 == 0           # letters disappear in pairs
    print(f"{name:20s} len={len(s):>6} -> len={len(result):>6} in {time.perf_counter() - start:.4f}s")

assert solution(cases["all A (even)"]) == ""
assert solution(cases["all A (odd)"]) == "A"
assert solution(cases["alternating ABAB..."]) == cases["alternating ABAB..."]

## Notes

- This is the "remove adjacent duplicates" pattern: a stack processes cancellations in one pass because each new letter can only interact with the most recent survivor.
- The uniqueness comes from the rewriting system being **confluent**: each letter is its own inverse and the result is the reduced word of the free product of three 2-element groups. You do not need that theory to solve the task.
- Repeated `str.replace("AA", "")` loops are tempting but O(N^2) in the worst case.